# 준비부터 받기까지 원판을 연속 몇 프레임 놓칠까?

베이스라인의 같은 예측을 신뢰도 0.25와 0.10에서 비교합니다. 실제 가림이 있는 검증 영상 `a03`과 비교 영상 `a05`를 사용합니다. 원본 영상에서 0부터 센 프레임 번호입니다.

두 영상의 기존 준비·받기 이벤트로 평가 범위를 먼저 정했습니다. 정답은 기존 원판 라벨을 우선 사용합니다. 라벨 사이의 빈 프레임은 두 기존 박스를 선형 보간한 뒤, 예측을 보지 않고 65장의 원본을 한 장씩 보며 보정했습니다. 평가용 추가 좌표는 별도 CSV에 원본 픽셀 단위로 보관합니다. `a03`의 시작은 기존 부분 가림 라벨처럼 랙 위로 드러난 원판 부분을 기준으로 삼았습니다. 모든 셀은 Mac의 프로젝트 커널에서 위에서부터 실행합니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle

from wlcv.config import ANNOTATIONS, ROOT, RUNS, VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.temporal import evaluate_frames, compare_thresholds

video_ids = ["a03_sn_f_01", "a05_sn_f_01"]
weights = RUNS / "plate_baseline/weights/best.pt"
review_csv = ANNOTATIONS / "temporal_review_boxes.csv"
thresholds = (0.25, 0.10)
out = RUNS / "plate_baseline/report/temporal_misses"
out.mkdir(parents=True, exist_ok=True)

## 1. 원본에서 평가용 박스 확인

기존 라벨 사이를 채운 프레임입니다. 하늘색 박스가 수행자가 들어 올리는 가까운 원판과 맞는지 먼저 확인하세요.

In [ ]:
def draw_box(axis, box, color, style="-"):
    x1, y1, x2, y2 = box
    axis.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                             edgecolor=color, linewidth=2, linestyle=style))

review = pd.read_csv(review_csv).set_index(["video_id", "frame_index"])
fig, axes = plt.subplots(1, 2, figsize=(6, 5), layout="constrained")
for axis, video in zip(axes, video_ids):
    frame = 12
    axis.imshow(read_rgb(VIDEOS / f"{video}.mp4", frame))
    draw_box(axis, review.loc[(video, frame)].tolist(), "#46deff")
    axis.set_title(f"{video}  |  f{frame}")
    axis.axis("off")
plt.show()

## 2. 프레임별 후보를 저장하고 연속 미탐 세기

원본 MP4를 매 프레임 디코딩합니다. CPU, 입력 크기 640, 후보 신뢰도 하한 0.001로 한 번 추론합니다. 가까운 원판 정답과 IoU가 0.50 이상인 후보 중 최고 신뢰도가 기준을 통과하면 정탐입니다. 정탐이 나오면 그 앞의 연속 미탐 구간이 끝납니다. 추가 박스는 기준을 통과한 박스 중 정탐 하나를 제외한 개수입니다.

`frames`는 평가한 장수, `missed`는 총 미탐, `longest_miss`는 최장 연속 미탐, `extra_box_frames`는 추가 박스가 한 개 이상 나온 프레임 수입니다. 두 평가 구간은 프레임 간 시간 간격이 약 1/24초입니다.

In [ ]:
frames, candidates = evaluate_frames(weights, review_csv, video_ids)
detail, intervals, summary = compare_thresholds(frames, candidates, thresholds)
timing = pd.read_csv(ROOT / "data/metadata/video_timing.csv").set_index("video_id")
fps = summary.video_id.map(timing.fps_from_median_gap)
summary["longest_seconds"] = (summary.longest_miss / fps).round(3)
for name, table in [("candidates", candidates), ("per_frame", detail),
                    ("miss_runs", intervals), ("summary", summary)]:
    table.to_csv(out / f"{name}.csv", index=False)
display(summary)
display(intervals)

## 3. 어느 구간에서 신뢰도가 떨어지는지 보기

선은 정답과 IoU가 0.50 이상인 후보의 최고 신뢰도입니다. 아래쪽 ×는 후보 하한 0.001에서도 매칭이 없었던 프레임입니다. 음영은 낮은 신뢰도 기준에서도 놓친 구간입니다.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True, sharey=True, layout="constrained")
for axis, video in zip(axes, video_ids):
    clip = detail[(detail.video_id == video) & (detail.threshold == thresholds[0])]
    axis.plot(clip.frame_index, clip.confidence, ".-", color="#176482", label="Matched confidence")
    missing = clip[clip.confidence.isna()]
    axis.scatter(missing.frame_index, [-0.045] * len(missing), marker="x", color="#202020",
                 label="No matching candidate" if len(missing) else None)
    missed = intervals[(intervals.video_id == video) & (intervals.threshold == min(thresholds))]
    for row in missed.itertuples():
        axis.axvspan(row.start - 0.5, row.end + 0.5, color="#eee0d3", zorder=0)
    for limit, style in zip(thresholds, ["--", ":"]):
        axis.axhline(limit, color="#88421d", linestyle=style, label=f"Threshold {limit:.2f}")
    axis.set(title=video, ylabel="Confidence", ylim=(-0.08, 1.04))
    axis.grid(axis="y", alpha=0.16)
axes[0].legend(loc="upper right", fontsize=8, ncol=2)
axes[-1].set_xlabel("Source video frame (0-based)")
fig.savefig(out / "confidence_timeline.png", dpi=150)
plt.show()

## 4. 가장 긴 미탐의 처음·중간·끝과 복구 장면 확인

`a03`의 2·8·14번은 첫 연속 미탐 구간, 15번은 다시 검출된 프레임입니다. 25번과 30번은 상승 중 점수가 낮은 사례입니다. 청록색 실선은 정답, 주황색 점선은 IoU 기준을 통과한 후보입니다. 낮은 신뢰도의 후보도 점선으로 표시하므로 각 그림의 점수를 함께 보세요.

In [ ]:
video = "a03_sn_f_01"
examples = [2, 8, 14, 15, 25, 30]
fig, axes = plt.subplots(2, 3, figsize=(9, 6), layout="constrained")
for axis, frame in zip(axes.flat, examples):
    rgb = read_rgb(VIDEOS / f"{video}.mp4", frame)
    truth = frames[(frames.video_id == video) & (frames.frame_index == frame)].iloc[0]
    axis.imshow(rgb)
    draw_box(axis, [truth.x1, truth.y1, truth.x2, truth.y2], "#46deff")
    matches = candidates[(candidates.video_id == video) & (candidates.frame_index == frame)
                         & (candidates.iou >= 0.50)]
    subtitle = "No matching candidate"
    if not matches.empty:
        box = matches.loc[matches.confidence.idxmax()]
        draw_box(axis, [box.x1, box.y1, box.x2, box.y2], "#ffad40", "--")
        subtitle = f"conf {box.confidence:.3f}   IoU {box.iou:.2f}"
    axis.set_xlim(max(0, truth.x1 - 75), min(rgb.shape[1], truth.x2 + 75))
    axis.set_ylim(min(rgb.shape[0], truth.y2 + 80), max(0, truth.y1 - 80))
    axis.set_title(f"f{frame}  |  {subtitle}", fontsize=10)
    axis.axis("off")
fig.suptitle(f"{video}   |   Ground truth (solid) and best matching candidate (dashed)")
fig.savefig(out / "a03_review.png", dpi=150)
plt.show()